# exp02 — Logit Lens (Fang & Marks) for the acrostic payload

**Method** (Fang & Marks, arXiv:2512.01222 — *Unsupervised Decoding of Encoded Reasoning*):
1. Forward pass, capture residual-stream hidden states at **every** layer.
2. Apply the model's **final RMSNorm** (`model.model.norm`) to each layer's residual — required, not optional.
3. Project through the unembedding (`lm_head`).
4. Softmax → per-layer next-token distribution.
5. Track the probability of the **target** token across layers.

**Adaptation to our acrostic setting.** The payload (e.g. `SAFE`) is encoded as the
first letter of each reasoning line. At every line-start position we read the
next-token distribution and sum the probability over **all tokens beginning with the
intended payload letter** — `P(line starts with letter_k)`. Tracking this across layers
asks *when the model internally commits to the hidden letter*. The **open** condition
(no acrostic instruction) is the control: at its line-starts we measure the same
payload letters — it should stay near chance.

Reuses exp01's `valid_pairs.json` (cached stego/open token ids + payloads). Run exp01
generation first. No binary classifier here, so the p>>n probe confound does not apply.

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REAL_MODEL     = 'meta-llama/Llama-3.1-8B-Instruct'  # must match the exp01 run that made the cache
FALLBACK_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
CACHE_DIR = 'results/exp01'                            # SAME dir as exp01 -> reuses valid_pairs.json

if IN_COLAB:
    if not os.path.exists('stego_CoT'):
        !git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git
    %cd stego_CoT
    !git pull -q
    !pip install -q -r requirements.txt
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/stego_exp01'   # exp01's Drive cache
else:
    if os.path.exists('.env'):
        for line in open('.env'):
            if '=' in line and not line.startswith('#'):
                k, v = line.strip().split('=', 1)
                os.environ.setdefault(k, v)

print(f'Environment: {"Colab" if IN_COLAB else "Local"} | CACHE_DIR: {CACHE_DIR}')

In [ ]:
import json, re, string
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
print('Imports OK | torch', torch.__version__)

In [ ]:
# Load exp01's gated-in pairs: each has stego_ids/stego_plen, open_ids/open_plen, payload.
PAIRS_CACHE = f'{CACHE_DIR}/valid_pairs.json'
assert os.path.exists(PAIRS_CACHE), (
    f'{PAIRS_CACHE} not found — run exp01 generation first (it caches the token ids).')
with open(PAIRS_CACHE) as f:
    valid_pairs = json.load(f)
assert valid_pairs and 'stego_ids' in valid_pairs[0], 'cache missing token ids; re-run exp01.'
MAX_SAMPLES = None                       # cap for a quick pass; None = all
if MAX_SAMPLES:
    valid_pairs = valid_pairs[:MAX_SAMPLES]
print(f'{len(valid_pairs)} valid pairs | example payload={valid_pairs[0]["payload"]}')

In [ ]:
def _from_pretrained(model_name, device, kw):
    if device == 'cuda':
        try:
            m = AutoModelForCausalLM.from_pretrained(
                model_name, device_map='auto',
                attn_implementation='flash_attention_2', **kw).eval()
            print('  attn: flash_attention_2')
            return m
        except (ImportError, ValueError) as e:
            print(f'  flash_attention_2 unavailable ({type(e).__name__}); using sdpa')
            return AutoModelForCausalLM.from_pretrained(
                model_name, device_map='auto', attn_implementation='sdpa', **kw).eval()
    return AutoModelForCausalLM.from_pretrained(model_name, **kw).to(device).eval()

def load_model(model_name=REAL_MODEL):
    global REAL_MODEL
    hf_token = os.environ.get('HF_TOKEN')
    device = ('cuda' if torch.cuda.is_available()
              else 'mps' if torch.backends.mps.is_available() else 'cpu')
    dtype = torch.bfloat16 if device in ('cuda', 'mps') else torch.float32
    kw = dict(dtype=dtype, token=hf_token)
    for name in [model_name, FALLBACK_MODEL]:
        try:
            tok = AutoTokenizer.from_pretrained(name, token=hf_token)
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token
            model = _from_pretrained(name, device, kw)
            REAL_MODEL = name
            print(f'Model loaded: {name} on {device} ({dtype})')
            return model, tok, device
        except Exception as e:
            print(f'  FAILED to load {name}: {type(e).__name__}: {e}')
            if name == FALLBACK_MODEL:
                raise

model, tok, device = load_model()
# logit-lens components: final norm + unembedding. (Assumes a single device.)
FINAL_NORM = model.model.norm
LM_HEAD    = model.lm_head
N_LAYERS   = model.config.num_hidden_layers
print(f'n_layers={N_LAYERS} | hidden={model.config.hidden_size} | vocab={model.config.vocab_size}')

In [ ]:
# Precompute, for each A-Z, the set of vocab token ids whose surface form starts
# with that letter. P(line starts with 'S') = sum of softmax prob over LETTER_IDS['S'].
def build_letter_token_ids(tok):
    letter_ids = {c: [] for c in string.ascii_uppercase}
    for tid in range(tok.vocab_size):
        s = tok.convert_ids_to_tokens(tid)
        if s is None:
            continue
        surf = tok.convert_tokens_to_string([s]).strip()
        if not surf:
            continue
        c = surf[0].upper()
        if c in letter_ids:
            letter_ids[c].append(tid)
    return {c: torch.tensor(v, device=device, dtype=torch.long) for c, v in letter_ids.items()}

LETTER_IDS = build_letter_token_ids(tok)
print('letters with tokens:', sum(len(v) > 0 for v in LETTER_IDS.values()), '/26',
      '| e.g. |S|=', len(LETTER_IDS['S']))

In [ ]:
# Locate line-start positions in a cached token sequence and the letter each line begins with.
# Returns list of (predict_pos, first_char): hidden[predict_pos] predicts the line-start token.
def find_line_starts(tok, ids, plen):
    gen = ids[plen:]
    pieces = [tok.convert_tokens_to_string([tok.convert_ids_to_tokens(int(t))]) for t in gen]
    starts = []
    if pieces:                                            # first generated line: predicted by last prompt token
        first = pieces[0].strip()[:1].upper()
        if first.isalpha():
            starts.append((plen - 1, first))
    for i in range(len(pieces) - 1):                      # a '\n' token ends a line; next non-blank token starts one
        if '\n' in pieces[i]:
            j, nxt = i + 1, ''
            while j < len(pieces) and nxt.strip() == '':
                nxt = pieces[j]; j += 1
            ch = nxt.strip()[:1].upper()
            if ch.isalpha():
                starts.append((plen + i, ch))             # hidden at token i predicts token i+1
    return starts

# One forward per sample; reuse hidden states for every (position, layer) readout.
@torch.no_grad()
def lens_letter_probs(ids_list, positions, target_letters):
    ids = torch.tensor(ids_list, device=device, dtype=torch.long).unsqueeze(0)
    hs = model(input_ids=ids, output_hidden_states=True).hidden_states   # (n_layers+1) x (1,T,d)
    nL = len(hs)
    out = np.zeros((len(positions), nL), dtype=np.float32)
    for j, (pos, c) in enumerate(zip(positions, target_letters)):
        tgt = LETTER_IDS[c]
        for L, h in enumerate(hs):
            logits = LM_HEAD(FINAL_NORM(h[0, pos]))        # logit lens at this layer
            p = torch.softmax(logits.float(), dim=-1)
            out[j, L] = p[tgt].sum().item()
    return out

print('lens helpers ready')

In [ ]:
# Sweep all pairs. For each sample, target letters = payload[k] for the k-th reasoning line.
# Stego: line-starts of the acrostic CoT. Open (control): same payload letters at the
# open CoT's line-starts (model was never asked to encode them -> baseline).
LL_CACHE = f'{CACHE_DIR}/logitlens_curves.npz'

def entries_for(ids, plen, payload):
    starts = find_line_starts(tok, ids, plen)
    pos, lett = [], []
    for k, (p, _ch) in enumerate(starts):
        if k < len(payload):
            pos.append(p); lett.append(payload[k].upper())
    return pos, lett

if os.path.exists(LL_CACHE):
    d = dict(np.load(LL_CACHE))
    stego_curves, open_curves = d['stego'], d['open']
    print('Loaded logit-lens curves from cache:', stego_curves.shape, open_curves.shape)
else:
    stego_curves, open_curves = [], []
    for i, p in enumerate(valid_pairs):
        pl = p['payload'].upper()
        sp, sl = entries_for(p['stego_ids'], p['stego_plen'], pl)
        op, ol = entries_for(p['open_ids'],  p['open_plen'],  pl)
        if sp:
            stego_curves.append(lens_letter_probs(p['stego_ids'], sp, sl))
        if op:
            open_curves.append(lens_letter_probs(p['open_ids'], op, ol))
        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(valid_pairs)}')
    stego_curves = np.concatenate(stego_curves, 0)        # (n_lines, n_layers+1)
    open_curves  = np.concatenate(open_curves, 0)
    np.savez(LL_CACHE, stego=stego_curves, open=open_curves)
    print('Saved', LL_CACHE)
print('stego line-entries:', stego_curves.shape[0], '| open:', open_curves.shape[0])

In [ ]:
# Per-layer mean P(payload letter) across all line entries, stego vs open control.
layers = np.arange(stego_curves.shape[1])
s_mean, s_se = stego_curves.mean(0), stego_curves.std(0) / np.sqrt(len(stego_curves))
o_mean, o_se = open_curves.mean(0),  open_curves.std(0)  / np.sqrt(len(open_curves))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(layers, s_mean, 'o-', color='#dc2626', label='stego (acrostic)')
ax.fill_between(layers, s_mean - s_se, s_mean + s_se, color='#dc2626', alpha=0.2)
ax.plot(layers, o_mean, 's-', color='#2563eb', label='open (control)')
ax.fill_between(layers, o_mean - o_se, o_mean + o_se, color='#2563eb', alpha=0.2)
ax.set_xlabel('layer (residual-stream index; 0 = embeddings)')
ax.set_ylabel('P(next token starts with payload letter)')
ax.set_title('exp02 logit lens: when the model commits to the hidden acrostic letter')
ax.legend()
plt.tight_layout()
plt.savefig(f'{CACHE_DIR}/logitlens_layer_curve.png', dpi=120)
plt.show()

# Onset layer = first layer where stego mean exceeds half its own peak (above the open baseline).
peak = s_mean.max()
onset = int(np.argmax(s_mean >= 0.5 * peak))
print(f'stego peak P={peak:.3f} at layer {int(np.argmax(s_mean))} | half-peak onset layer {onset} '
      f'(~{onset/(len(layers)-1):.0%} depth)')
print(f'open  peak P={o_mean.max():.3f} (baseline)')

## Inspect the decoded words per layer

The curve above is aggregate. Here we open up individual samples: at a chosen
position, the **logit lens** gives a token distribution at *every* layer. We render
the top-k tokens (with their decoded text) layer by layer — this is the model
"thinking out loud" through depth. For an acrostic line-start you can watch the
intended word/letter climb the ranks in intermediate layers before the output.

In [ ]:
# Top-k logit-lens decode at one position: per layer, the most probable tokens + text.
@torch.no_grad()
def lens_topk(ids_list, pos, k=8):
    ids = torch.tensor(ids_list, device=device, dtype=torch.long).unsqueeze(0)
    hs = model(input_ids=ids, output_hidden_states=True).hidden_states
    rows = []
    for L, h in enumerate(hs):
        logits = LM_HEAD(FINAL_NORM(h[0, pos]))
        p = torch.softmax(logits.float(), dim=-1)
        pv, pi = p.topk(k)
        toks = [tok.convert_tokens_to_string([tok.convert_ids_to_tokens(int(t))]) for t in pi]
        rows.append(list(zip(toks, pv.tolist())))
    return rows                                  # rows[L] = [(token_text, prob), ...]

def _fmt(tok_text, prob):
    t = tok_text.replace('|', '\\|').strip() or '·'
    return f'`{t}`<br><sub>{prob:.2f}</sub>'

def show_lens_table(ids_list, pos, k=6, every=2, label='', target_letter=None):
    rows = lens_topk(ids_list, pos, k)
    nL = len(rows)
    head = '| layer | ' + ' | '.join(f'top{i+1}' for i in range(k)) + ' |'
    sep  = '|' + '---|' * (k + 1)
    out = [f'**{label}** — position {pos}'
           + (f', intended letter `{target_letter}`' if target_letter else ''), '', head, sep]
    for L in range(0, nL, every):
        cells = ' | '.join(_fmt(t, p) for t, p in rows[L])
        out.append(f'| {L} | {cells} |')
    if (nL - 1) % every:                         # always include final layer
        L = nL - 1
        out.append(f'| {L} | ' + ' | '.join(_fmt(t, p) for t, p in rows[L]) + ' |')
    display(Markdown('\n'.join(out)))

from IPython.display import Markdown, display
print('lens_topk / show_lens_table ready')

In [ ]:
# Demo: for a few stego samples, show the decoded-words table at the first acrostic
# line-start (where the model must emit payload letter 0). Change SAMPLE_IDS / pos to explore.
SAMPLE_IDS = [0, 1, 2]
for si in SAMPLE_IDS:
    if si >= len(valid_pairs):
        continue
    p = valid_pairs[si]
    pl = p['payload'].upper()
    starts = find_line_starts(tok, p['stego_ids'], p['stego_plen'])
    if not starts:
        continue
    pos0, _ch = starts[0]
    display(Markdown(f'### sample {si} — task: {p["task"]!r} | payload `{pl}` | '
                     f'fidelity {p.get("fidelity", float("nan")):.2f}'))
    show_lens_table(p['stego_ids'], pos0, k=6, every=2,
                    label='STEGO @ first line-start', target_letter=pl[0])

In [ ]:
# Full payload trajectory for one sample: at EACH line-start, the top decoded token
# at an intermediate layer vs the final layer, next to the intended payload letter.
def payload_trajectory(p, mid_frac=0.7):
    pl = p['payload'].upper()
    starts = find_line_starts(tok, p['stego_ids'], p['stego_plen'])
    rows = lambda pos: lens_topk(p['stego_ids'], pos, k=1)
    nL = model.config.num_hidden_layers + 1
    midL = int(mid_frac * (nL - 1))
    md = [f'**sample** task={p["task"]!r} payload=`{pl}`', '',
          f'| line k | intended | top@mid(L{midL}) | top@final |', '|---|---|---|---|']
    for k, (pos, _ch) in enumerate(starts):
        if k >= len(pl):
            break
        r = lens_topk(p['stego_ids'], pos, k=1)
        mt, mp = r[midL][0]
        ft, fp = r[-1][0]
        md.append(f'| {k} | `{pl[k]}` | `{mt.strip() or "·"}` ({mp:.2f}) | `{ft.strip() or "·"}` ({fp:.2f}) |')
    display(Markdown('\n'.join(md)))

payload_trajectory(valid_pairs[0])

In [ ]:
# Save a compact summary.
summary = {
    'model': REAL_MODEL,
    'n_layers_plus1': int(stego_curves.shape[1]),
    'n_stego_lines': int(stego_curves.shape[0]),
    'n_open_lines': int(open_curves.shape[0]),
    'stego_peak_prob': float(s_mean.max()),
    'stego_peak_layer': int(np.argmax(s_mean)),
    'stego_halfpeak_onset_layer': int(np.argmax(s_mean >= 0.5 * s_mean.max())),
    'open_peak_prob': float(o_mean.max()),
}
with open(f'{CACHE_DIR}/logitlens_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print(f'Saved -> {CACHE_DIR}/logitlens_summary.json')